# ChurnShield — Feature Engineering

**Objective:** Transform raw cleaned data into ML-ready features.  
**Input:** 'data/processed/telco_cleaned.csv' 
**Output:** 'data/processed/telco_features.csv'

## Steps :
1. Load cleaned data
2. Drop irrelevant features
3. Encode categorical variables
4. Create new features
5. Scale numerical features
6. Export ML-ready dataset

In [7]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import LabelEncoder, StandardScaler

pd.set_option('display.max_columns', None)

# Load cleaned dataset from EDA step
df = pd.read_csv('../data/processed/telco_cleaned.csv')
print(f"✅ Dataset loaded — Shape: {df.shape}")
df.head()

✅ Dataset loaded — Shape: (7032, 21)


,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,Yes,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,No,Yes,No,No,No,One year,No,Mailed check,56.95,1889.50,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,Yes,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,No,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,No,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


In [8]:
# Drop irrelevant or redundant features identified during EDA
cols_to_drop = ['customerID', 'TotalCharges', 'gender', 'PhoneService']

df = df.drop(columns=cols_to_drop)

print("✅ Dropped columns :", cols_to_drop)
print(f"   Remaining features : {df.shape[1]} columns")
print(f"   Columns : {list(df.columns)}")

✅ Dropped columns : ['customerID', 'TotalCharges', 'gender', 'PhoneService']
   Remaining features : 17 columns
   Columns : ['SeniorCitizen', 'Partner', 'Dependents', 'tenure', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod', 'MonthlyCharges', 'Churn']


In [9]:
# --- New Feature 1 : HasFamily ---
# A customer with a partner OR dependents is considered to have a family
# Family = stability = lower churn risk
df['HasFamily'] = (
    (df['Partner'] == 'Yes') | (df['Dependents'] == 'Yes')
).astype(int)

# --- New Feature 2 : EngagementScore ---
# Count the number of additional services subscribed
# More services = more engaged = less likely to churn
services = ['OnlineSecurity', 'OnlineBackup', 'DeviceProtection',
            'TechSupport', 'StreamingTV', 'StreamingMovies']

df['EngagementScore'] = df[services].apply(
    lambda row: sum(val == 'Yes' for val in row), axis=1
)

# --- New Feature 3 : ChargesPerMonth_vs_Tenure ---
# Average monthly charges relative to tenure
# High charges for short tenure = high churn risk
df['ChargePerTenure'] = (
    df['MonthlyCharges'] / (df['tenure'] + 1)
).round(2)

print("✅ New features created :")
print(f"   HasFamily        — sample : {df['HasFamily'].value_counts().to_dict()}")
print(f"   EngagementScore  — range  : {df['EngagementScore'].min()} to {df['EngagementScore'].max()}")
print(f"   ChargePerTenure  — mean   : {df['ChargePerTenure'].mean():.2f}")

✅ New features created :
   HasFamily        — sample : {1: 3752, 0: 3280}
   EngagementScore  — range  : 0 to 6
   ChargePerTenure  — mean   : 5.71


In [10]:
# --- Binary encoding : Yes/No → 1/0 ---
binary_cols = ['Partner', 'Dependents', 'PaperlessBilling', 'Churn']
for col in binary_cols:
    df[col] = (df[col] == 'Yes').astype(int)

# --- Ordinal encoding : ordered categories ---
df['Contract'] = df['Contract'].map({
    'Month-to-month': 0,
    'One year': 1,
    'Two year': 2
})

df['PaymentMethod'] = df['PaymentMethod'].map({
    'Electronic check': 0,
    'Mailed check': 1,
    'Bank transfer (automatic)': 2,
    'Credit card (automatic)': 3
})

# --- One-Hot Encoding : non-ordered categories ---
df = pd.get_dummies(df, columns=['InternetService', 'MultipleLines',
                                  'OnlineSecurity', 'OnlineBackup',
                                  'DeviceProtection', 'TechSupport',
                                  'StreamingTV', 'StreamingMovies'],
                    drop_first=True)

print(f"✅ Encoding done — Shape: {df.shape}")
print(f"   Columns : {list(df.columns)}")

✅ Encoding done — Shape: (7032, 28)
   Columns : ['SeniorCitizen', 'Partner', 'Dependents', 'tenure', 'Contract', 'PaperlessBilling', 'PaymentMethod', 'MonthlyCharges', 'Churn', 'HasFamily', 'EngagementScore', 'ChargePerTenure', 'InternetService_Fiber optic', 'InternetService_No', 'MultipleLines_No phone service', 'MultipleLines_Yes', 'OnlineSecurity_No internet service', 'OnlineSecurity_Yes', 'OnlineBackup_No internet service', 'OnlineBackup_Yes', 'DeviceProtection_No internet service', 'DeviceProtection_Yes', 'TechSupport_No internet service', 'TechSupport_Yes', 'StreamingTV_No internet service', 'StreamingTV_Yes', 'StreamingMovies_No internet service', 'StreamingMovies_Yes']


In [11]:
from sklearn.preprocessing import StandardScaler

# Columns to scale
cols_to_scale = ['tenure', 'MonthlyCharges',
                 'EngagementScore', 'ChargePerTenure']

scaler = StandardScaler()
df[cols_to_scale] = scaler.fit_transform(df[cols_to_scale])

print("✅ Scaling done")
print(df[cols_to_scale].describe().round(2))

✅ Scaling done
        tenure  MonthlyCharges  EngagementScore  ChargePerTenure
count  7032.00         7032.00          7032.00          7032.00
mean     -0.00            0.00            -0.00             0.00
std       1.00            1.00             1.00             1.00
min      -1.28           -1.55            -1.10            -0.64
25%      -0.95           -0.97            -1.10            -0.52
50%      -0.14            0.18            -0.02            -0.43
75%       0.92            0.83             0.52             0.02
max       1.61            1.79             2.15             5.31


In [12]:
# Final check
print("=== Final Dataset Overview ===")
print(f"Shape        : {df.shape}")
print(f"Missing values: {df.isnull().sum().sum()}")
print(f"Target distribution:\n{df['Churn'].value_counts()}")

# Export ML-ready dataset
df.to_csv('../data/processed/telco_features.csv', index=False)
print("\n✅ ML-ready dataset saved → data/processed/telco_features.csv")

=== Final Dataset Overview ===
Shape        : (7032, 28)
Missing values: 0
Target distribution:
Churn
0    5163
1    1869
Name: count, dtype: int64

✅ ML-ready dataset saved → data/processed/telco_features.csv


## Conclusions

The raw dataset has been transformed into a clean ML-ready feature set.

**Transformations applied:**
- Dropped 4 irrelevant/redundant columns : `customerID`, `TotalCharges`, `gender`, `PhoneService`
- Binary encoding applied to : `Partner`, `Dependents`, `PaperlessBilling`, `Churn`
- Ordinal encoding applied to : `Contract`, `PaymentMethod`
- One-Hot encoding applied to : `InternetService`, `MultipleLines`, `OnlineSecurity`,
`OnlineBackup`, `DeviceProtection`, `TechSupport`, `StreamingTV`, `StreamingMovies`
- 3 new features created : `HasFamily`, `EngagementScore`, `ChargePerTenure`
- StandardScaler applied to numerical columns

**Next step : Model Training & Evaluation**